# AloePri — Qwen3-8B obfusqué, interrogé depuis le notebook

Architecture : le **serveur local** (`aloepri_modal/notebook_server.py`) fait le codage (tokenize + permutation), l'interrogation de Modal (`/generate`, qui ne reçoit que des **IDs permutés**), puis le décodage (dépermutation + detokenize).

- Modal ne voit **jamais** de texte ni les clés (posture stricte du POC).
- Les clés (`obfuscation_keys.json`) et le tokenizer restent sur cette machine.

Exécuter la cellule suivante **une seule fois** (elle démarre le serveur local en arrière-plan), puis utiliser `ask()`. 

In [1]:
# --- Configuration (adapter les chemins si besoin) ---
KEYS_PATH = "/home/cmauceri/deepseek-harness-ws/artifacts/obfuscation_keys.json"
MODAL_URL = "https://mauceri--aloepri-qwen3-modal-serve.modal.run"
API_KEY_PATH = "/home/cmauceri/.aloepri-api-key"
PYTHON = "/home/cmauceri/deepseek-harness-ws/venv/bin/python"   # python du venv (torch/transformers/requests)
PORT = 8002

import subprocess, time, os

with open(API_KEY_PATH) as f:
    API_KEY = f.read().strip()

# Démarre le serveur local en arrière-plan (une seule fois)
import glob
repo = "/home/cmauceri/deepseek-harness-ws/Secretarius"
proc = subprocess.Popen(
    [PYTHON, f"{repo}/aloepri_modal/notebook_server.py",
     "--keys", KEYS_PATH, "--url", MODAL_URL,
     "--api-key", API_KEY, "--port", str(PORT)],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
print("serveur local démarré (PID", proc.pid, ")")

# Attend que le serveur local réponde
import requests
for _ in range(30):
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2).status_code == 200:
            print("serveur prêt : http://127.0.0.1:%d/ask" % PORT)
            break
    except Exception:
        time.sleep(0.5)

serveur local démarré (PID 126658 )
serveur prêt : http://127.0.0.1:8002/ask


In [2]:
import requests

def ask(prompt, system=None, max_new_tokens=300, timeout=600):
    """Envoie un prompt au serveur local → renvoie le texte de la réponse.

    Le premier appel après une pause déclenche le cold start de Modal
    (chargement des 16 Go) : compter ~1-3 min, d'où le timeout large.
    """
    body = {"prompt": prompt, "max_new_tokens": max_new_tokens}
    if system:
        body["system"] = system
    r = requests.post(f"http://127.0.0.1:{PORT}/ask", json=body,
                      timeout=timeout)
    r.raise_for_status()
    data = r.json()
    return data["result"], data.get("usage")

In [3]:
# --- Premier essai (le 1er appel peut prendre 1-3 min : cold start Modal) ---
texte, usage = ask("Quelle est la capitale de la France ?")
print(texte)
print("\n[usage]", usage)

La capitale de la France est **Paris**. 😊

[usage] {'prompt_tokens': 22, 'completion_tokens': 14, 'total_tokens': 36}


In [4]:
# --- Encore quelques essais ---
for p in [
    "What is 17 times 23 ?",
    "Write a haiku about the sea.",
    "Donne une recette simple de crêpes.",
]:
    print("\n=== ", p)
    texte, _ = ask(p, max_new_tokens=200)
    print(texte[:300])


===  What is 17 times 23 ?
17 times 23 is calculated as:

$$ 17 \times 23 = 391 $$

So, the answer is **391**.

===  Write a haiku about the sea.
Waves whisper and crash,  
Salt-kissed winds embrace the shore—  
Tides dance on.  

*Note: This haiku is structured in traditional Japanese style (5-7-5 syllables), with imagery related to the sea.*

===  Donne une recette simple de crêpes.
Voici une **recette simple et délicieuse pour des crêpes** (vous pouvez les appeler comme ça, c’est un jeu de mots !) :

---

### 🥞 **Recette des crêpes (ou crêpes comme on les appelle)**  
**pour 2 personnes**  

#### **Ingrédients :**
- 1 œuf  
- 100 g de farine (type blanche ou maïs, selon votre 


In [5]:
# --- Premier essai (le 1er appel peut prendre 1-3 min : cold start Modal) ---
texte, usage = ask("Jeanne a trois frères et trois soeurs combien son frère Robert a-t-il de soeurs ?")
print(texte)
print("\n[usage]", usage)

La question est un peu trompeuse, mais voici la réponse :

**Jeanne a trois frères et trois sœurs.**

Cela signifie qu'il y a **3 frères (dont elle)** et **3 sœurs (dont elle)**.

Mais pour savoir **combien de sœurs a Robert**, nous devons déterminer combien de sœurs il a.

### Voici les informations :
- Il y a **3 frères (dont Robert)**.
- Il y a **3 sœurs (dont Jeanne)**.
- Cela signifie que **il y a 3 sœurs (dont Jeanne)**.
- Donc, **Robert a 3 sœurs (Jeanne, Marie, Sophie)**.

### Réponse finale :
**Robert a 3 sœurs.** ✅

### Réponse : **3**.

[usage] {'prompt_tokens': 35, 'completion_tokens': 187, 'total_tokens': 222}


## Notes

- **Cold start** : le premier `ask()` après ~5 min d'inactivité relance le conteneur Modal (chargement des 16 Go depuis le Volume) — compter 1 à 3 min. Les appels suivants sont rapides (~quelques secondes).
- **Arrêter le serveur local** quand vous avez fini : `proc.kill()` dans la cellule de démarrage, ou tuer le processus `notebook_server.py`.
- **Posture stricte** : le serveur local connaît les clés ; Modal ne reçoit que des IDs permutés. Ne pas exposer le port 8002 hors de la machine.
- Si `requests` n'est pas installé dans le kernel du notebook : `pip install requests`.